# OOP 3: Docstrings and Type Hints

## Table of Contents

1. [Why Docstrings? Why Type-Hint?](#1)
2. [Type Hints: The Essentials](#2)
3. [Type Hints for Data Science Code](#3)
4. [When Type Hints Get Complicated: Use a Protocol](#4)
5. [Docstrings: Google Style](#5)
6. [Generating Documentation with help()](#6)
7. [Exercise: Document and Annotate the Dataset Class](#7)

---
## 1. Why Docstrings? Why Type-Hint? <a id="1"></a>

Three months from now, someone will call your function with a `float` where you expected a `pd.DataFrame`. That someone might be you.

**Type hints** tell the IDE (and `mypy`) about your intent before runtime. **Docstrings** explain *what* and *why*, the things type hints can't express.

Neither is optional in production code. Colleagues will judge you. Future you will judge you. 

We introduce these now, in notebook 03, so every class you write from here on has them baked in from the start.

> **Note on Protocols:** Notebook 06 covers the relationship between type hints, duck typing, and Protocols, the right tool when you want structural rather than hierarchical typing. This notebook covers the day-to-day mechanics.

---
## 2. Type Hints: The Essentials <a id="2"></a>

Type hints don't enforce types at runtime (unless you use Pydantic or run `mypy`), but they:
- Give IDEs enough information for correct autocomplete and inline error detection
- Let `mypy` catch type mismatches statically, before the code ever runs
- Serve as machine-readable documentation: It's more precise than a comment

In [ ]:
from typing import Any

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Without type hints: the IDE has no idea what these are
def train_model_bad(data, config, features):
    pass

# With type hints: IDE, mypy, and colleagues all know exactly what's expected
def train_model_good(
    data: pd.DataFrame,
    target_column: str,
    feature_columns: list[str],
    n_estimators: int = 100,
    random_state: int | None = None,       # int or None (Python 3.10+ shorthand)
) -> dict[str, object]:
    """Train a random forest and return evaluation metrics."""

    X = data[feature_columns]
    y = data[target_column]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=random_state)
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
    clf.fit(X_tr, y_tr)
    return {"accuracy": accuracy_score(y_te, clf.predict(X_te)), "model": clf}


np.random.seed(42)
df = pd.DataFrame({
    "tenure":  np.random.randint(1, 60, 100),
    "charges": np.random.uniform(20, 100, 100),
    "churn":   (np.random.rand(100) > 0.7).astype(int),
})
result = train_model_good(df, "churn", ["tenure", "charges"], n_estimators=50, random_state=42)
print(f"Accuracy: {result['accuracy']:.3f}")

In [ ]:
# The main patterns you'll write every day

def common_patterns(
    x: int,                                 # basic built-in type
    name: str = "default",                  # with default value
    value: float | None = None,             # optional value  (3.10+ style)
    items: list[str] = None,                # list of strings
    mapping: dict[str, Any] = None,         # dict with string keys
    data: pd.DataFrame | np.ndarray = None, # either type
) -> tuple[np.ndarray, float]:              # fixed-length tuple return
    pass

# Older typing module syntax — you'll see this in existing codebases
from typing import Optional, Union, Tuple, List, Dict

def legacy_style(
    value: Optional[int] = None,          # same as int | None
    data: Union[pd.DataFrame, np.ndarray] = None,  # same as DataFrame | ndarray
) -> Tuple[np.ndarray, float]:            # same as tuple[ndarray, float]
    pass

print("Prefer the 3.10+ shorthand in new code, recognise the older style in existing code.")
print()
print("Annotations are stored on the function object:")
print(train_model_good.__annotations__)

---
## 3. Type Hints for Data Science Code <a id="3"></a>

A few DS-specific patterns that are worth knowing explicitly.

In [ ]:
# ── Callable: type-hint a function argument ───────────────────────────────
from collections.abc import Callable
from sklearn.metrics import accuracy_score, f1_score

def evaluate(
    y_true: pd.Series,
    y_pred: np.ndarray,
    metric: Callable[[pd.Series, np.ndarray], float] = accuracy_score,
) -> float:
    """Evaluate predictions using any sklearn-compatible metric."""
    return metric(y_true, y_pred)

np.random.seed(0)
y_t = pd.Series(np.random.randint(0, 2, 50))
y_p = np.random.randint(0, 2, 50)
print(f"accuracy : {evaluate(y_t, y_p):.3f}")
print(f"f1       : {evaluate(y_t, y_p, metric=f1_score):.3f}")

In [ ]:
# ── TypeAlias: name a complex type so you write it once ───────────────────
from typing import TypeAlias

MetricsDict: TypeAlias = dict[str, list[float]]
FeatureMatrix: TypeAlias = pd.DataFrame
TargetVector: TypeAlias = pd.Series

def record_metrics(run_id: str, metrics: MetricsDict) -> None:
    for name, values in metrics.items():
        print(f"  {run_id} | {name}: {sum(values)/len(values):.4f}")

record_metrics("run_001", {"accuracy": [0.87, 0.89, 0.88], "auc": [0.91, 0.93, 0.92]})

In [ ]:
# ── TYPE_CHECKING: avoid importing a heavy module just for a type hint ─────
from __future__ import annotations   # makes all annotations lazy strings by default
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from sklearn.pipeline import Pipeline  # only imported when mypy runs, not at runtime

def wrap_pipeline(pipeline: Pipeline, name: str) -> dict[str, object]:
    """Wrap a fitted pipeline with metadata."""
    return {"name": name, "steps": len(pipeline.steps)}

print("TYPE_CHECKING imports have zero runtime cost.")
print("Use them when a type hint would require importing a large library.")

---
## 4. When Type Hints Get Complicated: Use a Protocol <a id="4"></a>

The most common frustration with type hints in DS code is this:

```python
def train(model: BaseEstimator, X: pd.DataFrame, y: pd.Series) -> None:
    model.fit(X, y)
```

This works with sklearn models, but mypy rejects your own classes unless they inherit from `BaseEstimator`. That's the wrong constraint: you want *anything with a `.fit()` method*, regardless of where it comes from.

The solution is a **Protocol** (covered in depth in notebook 06):

```python
from typing import Protocol

class Estimator(Protocol):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "Estimator": ...
    def predict(self, X: pd.DataFrame) -> np.ndarray: ...

def train(model: Estimator, X: pd.DataFrame, y: pd.Series) -> None:
    model.fit(X, y)
```

Now `train()` accepts sklearn models, your own models, and any future model, as long as they have the right methods. mypy checks the signatures, not the inheritance chain.

**Rule of thumb:**
- Use class names as type hints for *your own* hierarchy
- Use Protocols when the type hint needs to describe an interface that others already satisfy

---
## 5. Docstrings: Google Style <a id="5"></a>

There are several docstring conventions (NumPy, reStructuredText, Google). **Google style** is the most readable in source and supported by most documentation generators.

Rule: document `Args`, `Returns`, `Raises`, and anything non-obvious. Don't re-state what's already clear from the type hint.

In [ ]:
class FeatureSelector:
    """Selects features based on importance scores from a fitted tree model.

    Useful as a preprocessing step to remove low-signal features before
    training a final model.

    Attributes:
        threshold: Minimum importance score required to keep a feature.
        selected_features_: Feature names selected after fitting.
            Trailing underscore indicates this is set by fit(), not __init__().

    Example:
        > selector = FeatureSelector(threshold=0.05)
        > selector.fit(X_train, y_train)
        > X_reduced = selector.transform(X_train)
    """

    def __init__(self, threshold: float = 0.01) -> None:
        """Initialise the selector.

        Args:
            threshold: Features with importance below this value are dropped.
                Must be in (0, 1]. Defaults to 0.01.
        """
        self.threshold = threshold
        self.selected_features_: list[str] = []

    def fit(self, X: pd.DataFrame, y: pd.Series) -> FeatureSelector:
        """Fit a Random Forest and select features above the threshold.

        Args:
            X: Feature matrix. Must not contain NaN values.
            y: Target labels, aligned with X.

        Returns:
            self, to allow method chaining.

        Raises:
            ValueError: If the threshold eliminates all features.
        """
        from sklearn.ensemble import RandomForestClassifier

        rf = RandomForestClassifier(n_estimators=50, random_state=42)
        rf.fit(X, y)
        importances = pd.Series(rf.feature_importances_, index=X.columns)
        self.selected_features_ = importances[importances >= self.threshold].index.tolist()

        if not self.selected_features_:
            raise ValueError(
                f"threshold={self.threshold} eliminates all {len(X.columns)} features."
            )
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Return only the selected feature columns.

        Args:
            X: Feature matrix with the same columns as the fitting data.

        Returns:
            DataFrame containing only the selected columns.

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if not self.selected_features_:
            raise RuntimeError("Call fit() before transform().")
        return X[self.selected_features_]


np.random.seed(42)
X_demo = pd.DataFrame({
    "tenure":   np.random.randint(1, 60, 200),
    "charges":  np.random.uniform(20, 100, 200),
    "noise_1":  np.random.randn(200),
    "noise_2":  np.random.randn(200),
})
y_demo = pd.Series((X_demo["tenure"] < 15).astype(int))

selector = FeatureSelector(threshold=0.10)
selector.fit(X_demo, y_demo)
print("Selected:", selector.selected_features_)
print(selector.transform(X_demo).head(3))

---
## 6. Generating Documentation with help() <a id="6"></a>

`help()` renders your docstrings immediately in the REPL or Jupyter. VS Code and Jupyter show them on hover. This is the free documentation your docstrings buy you.

In [ ]:
help(FeatureSelector.fit)

In [ ]:
# Access docstrings programmatically
print(FeatureSelector.__doc__[:300], "...")

---
## 7. Exercise: Document and Annotate the Dataset Class <a id="7"></a>

Take the `Dataset` class from notebook 01 (reproduced below without docstrings or complete type hints) and add:

1. A class-level docstring explaining what it is and when to use it
2. Google-style docstrings for `__init__`, `split`, `summary`, and `class_balance`
3. Complete type hints on all methods — including return types

When you're done, run `help(Dataset.split)` to verify it renders correctly.

In [ ]:
from sklearn.model_selection import train_test_split


class Dataset:

    def __init__(self, X, y, name="dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits = {}

    def split(self, test_size=0.2, random_state=42):
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self):
        return self._splits["X_train"], self._splits["y_train"]

    @property
    def test(self):
        return self._splits["X_test"], self._splits["y_test"]

    @property
    def feature_names(self):
        return self.X.columns.tolist()

    def summary(self):
        n, p = self.X.shape
        pos = int(self.y.sum())
        return f"Dataset('{self.name}'): {n} rows x {p} features, {pos}/{n} positive"

    def class_balance(self):
        return self.y.value_counts(normalize=True).rename("proportion")


# Verify
help(Dataset.split)

Do you need a bit of help? Here you can find a possible solution.

> <details>
> <summary>Click here</summary>
>
> ```python
> class Dataset:
>     """A container for a supervised learning dataset with train/test splitting.
>
>     Bundles X, y, feature names, and split partitions into one object,
>     replacing the loose variables (X_train, X_test, y_train, ...) that
>     typically accumulate in a notebook.
>     """
>
>     def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset") -> None:
>         """Initialise the dataset.
>
>         Args:
>             X: Feature matrix.
>             y: Target labels, aligned with X.
>             name: Human-readable identifier used in summaries.
>         """
>         self.X = X
>         self.y = y
>         self.name = name
>         self._splits: dict = {}
>
>     def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
>         """Partition into train and test sets.
>
>         Args:
>             test_size: Proportion of rows to hold out for testing.
>             random_state: Seed for reproducibility.
>
>         Returns:
>             self, to allow method chaining.
>         """
>         ...
>
>     def summary(self) -> str:
>         """Return a one-line description of the dataset.
>
>         Returns:
>             String with row count, feature count, and positive rate.
>         """
>         ...
>
>     def class_balance(self) -> pd.Series:
>         """Return the proportion of each class label.
>
>         Returns:
>             Series indexed by class label, values are proportions summing to 1.
>         """
>         ...
> ```
> </details>